# Chapter 3 — The Reveal: Which Communities Are Being Left Behind?

---

## The Story

Margaret is 84 years old and lives in Central Highlands, Tasmania. Her GP has assessed her as Level 4 — the highest home care need. She qualifies for a residential aged care bed. But there are only 18 beds in her entire region, and 56 people ahead of her with the same urgent need.

**Her waitlist ratio: 3.1 — meaning for every 1 bed, 3 people are waiting.**

Margaret's story is not unusual. Across Australia, 58 SA3 regions are in the same situation: demand structurally exceeds supply. This chapter names them.

---

## Narrative Arc — The Detective: Resolution

Ch 1 showed the map of where the gap is. Ch 2 showed why quality differs (ownership). **Ch 3 is the resolution** — the specific communities where the gap is not just statistical, it is a lived crisis.

**Three questions this chapter answers:**
1. **Where?** — Which 20 SA3 regions have the worst supply–demand mismatch?
2. **Who?** — Who exactly is waiting, and how urgent is their need?
3. **Is it getting worse?** — Year-over-year: are more communities falling into deficit?

---

## User Stories

| Audience | User Story | Acceptance Criteria |
|----------|-----------|--------------------|
| Family caregiver | As a family member planning residential care for an elderly parent, I need to know which regions have the worst waitlists so I can make informed decisions about location | Dashboard shows ranked list with hover detail — specific numbers per region |
| Worker entering the sector | As someone entering the aged care workforce, I need to know where demand is highest so I can find job security | Map of regions in deficit clearly marked, with demand growth trend |
| Business / investor | As someone evaluating where to open or expand a facility, I need to see which regions have documented unmet demand so I can identify the highest-opportunity markets | Top 20 ranked by pressure with state and remoteness — cross-referenceable with supply data |

## Setup

In [26]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go

CLEAN = '../../data/clean'

# Load source files
users  = pd.read_csv(f'{CLEAN}/service_users_by_sa3.csv')
supply = pd.read_csv(f'{CLEAN}/service_supply_by_sa3.csv')
pop    = pd.read_csv(f'{CLEAN}/abs_population_by_sa3.csv')
rats   = pd.read_csv(f'{CLEAN}/star_ratings_by_facility.csv')

# Quality score + mmm_code per SA3 (mean quality across all snapshots, modal mmm_code)
quality_sa3 = (
    rats.groupby('sa3_code')
    .agg(
        quality_score=('quality_score', 'mean'),
        mmm_code=('mmm_code', lambda x: x.mode()[0] if x.notna().any() else np.nan)
    )
    .reset_index()
)

YEAR = 2024

def build_year_df(yr):
    """Join users + supply + pop + quality for a given year."""
    d = (
        users[users['year'] == yr]
        .merge(
            supply[supply['year'] == yr][['sa3_code', 'residential_places', 'n_residential']],
            on='sa3_code', how='left'
        )
        .merge(
            pop[pop['year'] == yr][['sa3_code', 'state', 'pop_65_plus']],
            on='sa3_code', how='left'
        )
        .merge(quality_sa3, on='sa3_code', how='left')
    )
    d = d[d['sa3_code'] != 10702]                       # Illawarra Catchment Reserve — pop_65_plus = 0
    d = d[d['residential_places'].fillna(0) > 0]        # need beds to compute pressure
    d['waitlist_pressure']       = d['hcp_high_needs'] / d['residential_places']
    d['access_rate_residential'] = d['total_residential'] / d['pop_65_plus'] * 100
    d['access_rate_homecare']    = d['total_homecare']    / d['pop_65_plus'] * 100
    d['access_rate_combined']    = (d['total_residential'] + d['total_homecare']) / d['pop_65_plus'] * 100
    d['access_rate_hcp_high']    = d['hcp_high_needs']    / d['pop_65_plus'] * 100  # L3+L4 hidden demand
    return d

df = build_year_df(YEAR)

# Pre-compute key narrative numbers
n_over_1  = (df['waitlist_pressure'] > 1.0).sum()
n_over_2  = (df['waitlist_pressure'] > 2.0).sum()
med_wp    = df['waitlist_pressure'].median()
worst_sa3 = df.nlargest(1, 'waitlist_pressure').iloc[0]

print(f'Year: {YEAR}')
print(f'Communities where demand exceeds supply (pressure > 1.0): {n_over_1}')
print(f'Communities where demand is DOUBLE supply (pressure > 2.0): {n_over_2}')
print(f'National median pressure: {med_wp:.3f}')
print(f'Worst region: {worst_sa3["sa3_name"]} ({worst_sa3["state"]}) = {worst_sa3["waitlist_pressure"]:.3f}')
print()
print(f'National median access rates (2024):')
print(f'  Residential:  {df["access_rate_residential"].median():.2f}%')
print(f'  Home care:    {df["access_rate_homecare"].median():.2f}%')
print(f'  Combined:     {df["access_rate_combined"].median():.2f}%')
print(f'  HCP high (L3+L4 in home care): {df["access_rate_hcp_high"].median():.2f}%')

Year: 2024
Communities where demand exceeds supply (pressure > 1.0): 58
Communities where demand is DOUBLE supply (pressure > 2.0): 8
National median pressure: 0.636
Worst region: Central Highlands (Tas.) (TAS) = 3.111

National median access rates (2024):
  Residential:  4.02%
  Home care:    5.23%
  Combined:     9.43%
  HCP high (L3+L4 in home care): 2.90%


### What this tells us

The national picture is stark before we even look at individual communities. **58 SA3 regions** — nearly 1 in 6 of all analysed areas — have more high-needs home care users than available residential beds. **8 of those have double the demand.** The national median of 0.636 means that even in a typical region, for every 10 beds there are already 6 people with high-level needs in home care who could qualify for residential placement. Central Highlands, Tasmania holds the worst ratio in the country at 3.111 — more than three high-needs people for every single available bed.

## What: The Scale of the Problem

Before naming specific communities, we need to understand the national picture.

The home care system was designed to keep people in their homes with support. But it is increasingly being used as a **holding pattern** — people approved for L3 and L4 care (near-residential intensity) who cannot access a residential bed, so they stay at home with ever-increasing care needs.

**What L3 and L4 actually means:**
- **L3 (High):** Requires significant daily assistance — help with bathing, dressing, medication management, meal preparation. Roughly equivalent to 3–4 hours of care per day.
- **L4 (Very High):** Requires near-constant support — dementia care, complex wound management, mobility assistance. These are people who, in most clinical assessments, belong in residential care.

The chart below shows how the composition of home care has shifted since 2023.

In [27]:
nat = users.groupby('year')[[
    'hcp_level1','hcp_level2','hcp_level3','hcp_level4','total_homecare'
]].sum().reset_index()

nat_melt = nat.melt(
    id_vars='year',
    value_vars=['hcp_level1','hcp_level2','hcp_level3','hcp_level4'],
    var_name='level', value_name='users'
)
nat_melt['level_label'] = nat_melt['level'].map({
    'hcp_level1': 'L1 — Basic daily support',
    'hcp_level2': 'L2 — Moderate support',
    'hcp_level3': 'L3 — High (near-residential)',
    'hcp_level4': 'L4 — Very High (should be in residential care)'
})

hcp_2023 = users[users['year']==2023]['hcp_high_needs'].sum()
hcp_2025 = users[users['year']==2025]['hcp_high_needs'].sum()
pct_2025 = users[users['year']==2025]['hcp_high_needs'].sum() / users[users['year']==2025]['total_homecare'].sum() * 100

fig = px.area(
    nat_melt,
    x='year', y='users', color='level_label',
    category_orders={'level_label': [
        'L1 — Basic daily support',
        'L2 — Moderate support',
        'L3 — High (near-residential)',
        'L4 — Very High (should be in residential care)'
    ]},
    color_discrete_map={
        'L1 — Basic daily support':                   '#a6c8ff',
        'L2 — Moderate support':                      '#4589ff',
        'L3 — High (near-residential)':               '#ff832b',
        'L4 — Very High (should be in residential care)': '#da1e28'
    },
    title='Australia\'s home care system is carrying people who need residential care<br>'
          f'<sup>L3+L4 users grew from {hcp_2023:,.0f} (2023) to {hcp_2025:,.0f} (2025) — '
          f'+22% in 2 years. Now {pct_2025:.0f}% of all home care approvals.</sup>',
    labels={'users': 'People receiving home care', 'year': '', 'level_label': 'Care level'}
)
fig.update_layout(height=400)
fig.show()

print(f'Key number: {pct_2025:.1f}% of all home care approvals are now L3 or L4 (2025).')
print(f'These {hcp_2025:,.0f} people need residential care — but the beds are not there.')

Key number: 59.1% of all home care approvals are now L3 or L4 (2025).
These 172,285 people need residential care — but the beds are not there.


### What this chart reveals

The home care system was designed to help people live independently at home with light support. The stacked area chart shows it has become something fundamentally different. **59.1% of all home care approvals in 2025 are now L3 or L4** — the two highest care levels, representing people with complex, intensive needs who in most clinical assessments belong in residential care.

The orange and red layers (L3 and L4) are visibly growing relative to the blue layers (L1 and L2). This is not a temporary spike — it is a structural shift driven by an ageing population accumulating in the home care system because residential beds are not being built fast enough. The 172,285 people at L3+L4 in 2025 represent a +22% increase from 140,968 in just two years.

## So What: These Are the Communities Left Behind

**`waitlist_pressure` = high-needs HCP users (L3+L4) ÷ available residential beds**

A pressure of 1.0 means demand exactly equals supply. Every region above 1.0 is in structural deficit — there are more people who need beds than beds available.

**The reference line at 1.0 is the line between manageable and crisis.**

Colour shows remoteness (MMM). The pattern is striking: this is not purely a remote problem. The majority of the top 20 are MM1 — major cities — where population density creates concentrated demand that supply has not kept pace with.

In [28]:
top20 = (
    df.nlargest(30, 'waitlist_pressure')
    .drop_duplicates('sa3_name')
    .head(20)
    .sort_values('waitlist_pressure')
)

MMM_COLOURS = {
    'MM1':'#1f77b4','MM2':'#ff7f0e','MM3':'#2ca02c',
    'MM4':'#d62728','MM5':'#9467bd','MM6':'#8c564b','MM7':'#e377c2'
}

fig = px.bar(
    top20,
    x='waitlist_pressure', y='sa3_name',
    color='mmm_code',
    color_discrete_map=MMM_COLOURS,
    orientation='h',
    hover_data={
        'state': True,
        'hcp_high_needs': ':,d',
        'residential_places': ':,d',
        'waitlist_pressure': ':.2f',
        'pop_65_plus': ':,.0f'
    },
    title='20 communities where demand has outrun supply — 2024<br>'
          '<sup>Every bar past the red line represents a community in structural deficit. '
          'These are real places where real people cannot get a bed.</sup>',
    labels={
        'waitlist_pressure': 'High-needs home care users per residential bed',
        'sa3_name': '',
        'mmm_code': 'Remoteness'
    }
)
fig.add_vline(
    x=1.0, line_dash='dash', line_color='red', line_width=2,
    annotation_text='Demand = Supply',
    annotation_position='top right'
)
fig.update_layout(height=540)
fig.show()

# So what statement
worst = top20.iloc[-1]
second = top20.iloc[-2]
mm1_count = (top20['mmm_code'] == 'MM1').sum()
print(f'So what: {worst["sa3_name"]} ({worst["state"]}) has {worst["waitlist_pressure"]:.1f} '
      f'high-needs people per bed.')
print(f'  For a family there: finding a residential bed is not difficult — it is effectively impossible.')
print(f'  {second["sa3_name"]} is second at {second["waitlist_pressure"]:.2f} — '
      f'{int(second["hcp_high_needs"])} people, {int(second["residential_places"])} beds.')
print(f'  {mm1_count} of the top 20 are major cities (MM1) — this is not a remote problem.')

So what: Central Highlands (Tas.) (TAS) has 3.1 high-needs people per bed.
  For a family there: finding a residential bed is not difficult — it is effectively impossible.
  Noosa Hinterland is second at 2.83 — 255 people, 90 beds.
  12 of the top 20 are major cities (MM1) — this is not a remote problem.


### What this chart reveals

Every bar that extends past the red dashed line represents a community in structural deficit. The line is not an aspirational target — it is the point where one available bed exists for every high-needs person waiting. Anything beyond it means the system has already failed to keep pace with demand.

**Central Highlands (TAS) at 3.1** is the most extreme case nationally: 56 people competing for 18 beds. For a family in this region, the wait is not measured in weeks — it is effectively indefinite. **Noosa Hinterland (QLD) at 2.83** is second: 255 people, 90 beds.

The colour pattern challenges the assumption that this is a remote problem. **12 of the top 20 are MM1 major cities** — inner and outer suburban areas where population density has created concentrated demand that supply has not followed. The crisis is geographic, not demographic — it is about where beds are, not who needs them.

## Three-Tier Access Gap: Why These Regions Are Under-Served

`waitlist_pressure` shows *how stretched* supply is relative to demand. But it doesn't explain *why* some regions end up with so little supply. Three access rate dimensions make the structural pattern visible:

| Metric | Formula | What it shows |
|--------|---------|---------------|
| `access_rate_residential` | total_residential / pop_65_plus × 100 | Who is actually IN residential care today |
| `access_rate_hcp_high` | hcp_high_needs / pop_65_plus × 100 | L3+L4 home care users per elderly population — hidden residential demand |
| `access_rate_combined` | (total_residential + total_homecare) / pop_65_plus × 100 | Total aged care reach of the system |

**The key signal:** when `access_rate_hcp_high` exceeds `access_rate_residential`, the system has more people who need residential beds than it has beds to offer. Home care is functioning as a holding pen — not by design, but by default.

National median (2024): residential **4.02%** vs HCP high-needs **2.90%**. In crisis regions, this ratio flips.

In [29]:
top20_sorted = top20.sort_values('waitlist_pressure')

fig = go.Figure()

fig.add_trace(go.Bar(
    y=top20_sorted['sa3_name'],
    x=top20_sorted['access_rate_residential'],
    name='In residential care',
    orientation='h',
    marker_color='#4C78A8',
    hovertemplate='%{y}<br>Residential access: %{x:.2f}% of pop 65+<extra></extra>',
))

fig.add_trace(go.Bar(
    y=top20_sorted['sa3_name'],
    x=top20_sorted['access_rate_hcp_high'],
    name='L3+L4 in home care (need residential)',
    orientation='h',
    marker_color='#da1e28',
    hovertemplate='%{y}<br>Hidden demand (L3+L4): %{x:.2f}% of pop 65+<extra></extra>',
))

fig.update_layout(
    barmode='group',
    title='Supply vs hidden demand — per 100 elderly residents, top 20 pressure regions 2024<br>'
          '<sup>When red exceeds blue: more people need residential beds than can access them. '
          'National median residential: 4.02%, national median HCP high: 2.90%</sup>',
    xaxis_title='People per 100 elderly residents (pop_65_plus)',
    yaxis_title='',
    height=560,
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1)
)
fig.show()

flipped = (top20['access_rate_hcp_high'] > top20['access_rate_residential']).sum()
nat_res = df['access_rate_residential'].median()
nat_hcp = df['access_rate_hcp_high'].median()
top20_res = top20['access_rate_residential'].mean()
top20_hcp = top20['access_rate_hcp_high'].mean()

print(f'{flipped}/20 worst regions: L3+L4 home care exceeds residential access per capita (ratio flipped)')
print(f'National median — residential: {nat_res:.2f}%  |  HCP high: {nat_hcp:.2f}%')
print(f'Top 20 average — residential: {top20_res:.2f}%  |  HCP high: {top20_hcp:.2f}%')
print(f'Top 20 residential is {nat_res/top20_res:.1f}x below national median — supply is the constraint, not demand.')

19/20 worst regions: L3+L4 home care exceeds residential access per capita (ratio flipped)
National median — residential: 4.02%  |  HCP high: 2.90%
Top 20 average — residential: 1.84%  |  HCP high: 4.46%
Top 20 residential is 2.2x below national median — supply is the constraint, not demand.


## Zooming In: Who Is Actually Waiting?

The bar chart above shows *how many* people are waiting relative to supply. This chart answers *who* they are.

In every one of these 20 communities, the dominant colours are orange and red — L3 and L4. These are not people with minor support needs who could be managed with home visits. They are people with complex, intensive care needs who have been assessed as requiring near-residential or residential-level support.

**The orange and red bars are not a queue — they are a crisis being absorbed invisibly into the home care system.**

In [30]:
hcp_melt = top20.melt(
    id_vars=['sa3_name','waitlist_pressure'],
    value_vars=['hcp_level1','hcp_level2','hcp_level3','hcp_level4'],
    var_name='level', value_name='users'
)
hcp_melt['level_label'] = hcp_melt['level'].map({
    'hcp_level1': 'L1 — Basic',
    'hcp_level2': 'L2 — Moderate',
    'hcp_level3': 'L3 — High (near-residential)',
    'hcp_level4': 'L4 — Very High (should be in residential)'
})

sa3_order = top20.sort_values('waitlist_pressure')['sa3_name'].tolist()

fig = px.bar(
    hcp_melt,
    x='sa3_name', y='users', color='level_label',
    category_orders={
        'sa3_name': sa3_order,
        'level_label': ['L1 — Basic','L2 — Moderate',
                        'L3 — High (near-residential)',
                        'L4 — Very High (should be in residential)']
    },
    color_discrete_map={
        'L1 — Basic':                         '#a6c8ff',
        'L2 — Moderate':                      '#4589ff',
        'L3 — High (near-residential)':       '#ff832b',
        'L4 — Very High (should be in residential)': '#da1e28'
    },
    barmode='stack',
    title='The people waiting — care levels in the 20 most pressured communities<br>'
          '<sup>Orange and red = people who need residential care, '
          'not just home support. In many regions this is the majority.</sup>',
    labels={'users':'People receiving home care','sa3_name':'','level_label':'Care level'}
)
fig.update_layout(height=440, xaxis_tickangle=45)
fig.show()

l34 = top20[['hcp_level3','hcp_level4']].sum().sum()
total = top20[['hcp_level1','hcp_level2','hcp_level3','hcp_level4']].sum().sum()
print(f'Across the top 20 communities: {l34/total*100:.1f}% of home care users are L3+L4.')
print(f'That is {int(l34):,} people receiving near-residential care at home')
print(f'because their community does not have enough beds to place them.')

Across the top 20 communities: 65.8% of home care users are L3+L4.
That is 10,086 people receiving near-residential care at home
because their community does not have enough beds to place them.


### What this chart reveals

The stacked bar answers the question the ranked list cannot: *who* is in the queue? In every one of these 20 communities, orange and red — L3 and L4 — dominate the stack. These are not people receiving a weekly shopping assist. They are people with dementia, complex wound care, full mobility dependence, or cognitive impairment severe enough that clinical assessors have categorised them as requiring near-residential-level support.

**Across the top 20 communities, 65.8% of all home care users are at L3+L4.** That is 10,086 people receiving near-residential care at home — against only 5,189 available residential beds, a combined ratio of 1.94x. The queue is not a line of people who would prefer home care. It is a queue of people who have exhausted what home care can safely provide.

## The Details: Every Community, Every Number

For families making a decision, for planners allocating funding, for workers choosing where to work — the ranked list above needs to be paired with the specific numbers.

Two things stand out in this table:

**Access rate and pressure do not always correlate.** Central Highlands TAS has a 0.64% access rate — very few people are actually in residential care — yet it has the worst pressure in Australia. The demand is real, the beds are simply not there. Surfers Paradise QLD is similar: 0.44% access rate, but 2.0 pressure.

**Some high-pressure regions also have lower quality.** The Hills District QLD (pressure 2.029) has a quality score of 2.969 — below standard. Families in these communities face both a waitlist and quality risk if they do get a place.

In [31]:
detail = top20.sort_values('waitlist_pressure', ascending=False)[[
    'sa3_name', 'state', 'mmm_code',
    'hcp_high_needs', 'residential_places', 'waitlist_pressure',
    'access_rate_residential', 'access_rate_hcp_high', 'quality_score', 'pop_65_plus'
]].rename(columns={
    'sa3_name':              'SA3',
    'state':                 'State',
    'mmm_code':              'MMM',
    'hcp_high_needs':        'HCP L3+L4',
    'residential_places':    'Beds',
    'waitlist_pressure':     'Pressure',
    'access_rate_residential': 'Residential %',
    'access_rate_hcp_high':  'HCP High %',
    'quality_score':         'Quality',
    'pop_65_plus':           'Pop 65+'
})
detail['Pressure']      = detail['Pressure'].round(3)
detail['Residential %'] = detail['Residential %'].round(2)
detail['HCP High %']    = detail['HCP High %'].round(2)
detail['Quality']       = detail['Quality'].round(3)
detail['Pop 65+']       = detail['Pop 65+'].astype(int)
print(detail.reset_index(drop=True).to_string(index=False))

                       SA3 State MMM  HCP L3+L4  Beds  Pressure  Residential %  HCP High %  Quality  Pop 65+
  Central Highlands (Tas.)   TAS MM5         56  18.0     3.111           0.64        1.98    4.521     2829
          Noosa Hinterland   QLD MM2        255  90.0     2.833           1.23        3.92    3.583     6505
 Sunshine Coast Hinterland   QLD MM1        659 252.0     2.615           1.72        4.62    3.685    14275
        Wheat Belt - North    WA MM4        964 369.0     2.612           0.89        6.57    3.406    14662
         Gympie - Cooloola   QLD MM3       1034 445.0     2.324           2.67        6.93    3.629    14930
                     Yarra   VIC MM1        618 270.0     2.289           2.13        5.36    3.972    11521
        The Hills District   QLD MM1        345 170.0     2.029           1.11        2.39    3.228    14423
                   Kwinana    WA MM1        249 123.0     2.024           2.22        4.80    3.312     5185
          Surfers P

### Reading the table

Two patterns emerge that the chart cannot show:

**Low access rate + high pressure = supply failure, not low demand.** Central Highlands TAS has a 0.64% access rate — almost no one is getting into residential care — yet pressure is 3.111. Surfers Paradise QLD is even more extreme: 0.44% access rate, pressure of 2.0. These are not communities where people choose home care over residential. They are communities where residential care is simply unavailable, so people accumulate in home care regardless of their clinical need.

**High pressure + low quality = compounded crisis.** The Hills District QLD (pressure 2.029, quality 2.969) and Kwinana WA (pressure 2.024, quality 3.188) face both problems simultaneously — families cannot get a bed, and the beds that exist are below standard. For these communities, the crisis is not just about quantity. It is about whether the available care is safe.

## What Next: Is It Getting Worse?

The map of crisis communities is not static. Between 2023 and 2024, **10 new SA3 regions crossed the pressure = 1.0 threshold** — communities where demand now structurally exceeds supply for the first time.

This matters because the trend is not cyclical. It is driven by demographics: Australia's 65+ population grew from 4.0M (2019) to 4.7M (2024), and this cohort is ageing further into high-dependency years. Supply cannot be built fast enough to keep pace with a demographic wave that was foreseeable decades ago.

**For families:** a region sitting at 0.8 pressure today may cross 1.0 within a year. The time to check your region and start planning is before crisis hits — not after.

In [32]:
m23 = build_year_df(2023)
m24 = build_year_df(2024)

# SA3s that crossed 1.0 threshold between 2023 and 2024
over1_2023 = set(m23[m23['waitlist_pressure'] > 1.0]['sa3_name'])
over1_2024 = set(m24[m24['waitlist_pressure'] > 1.0]['sa3_name'])
new_crisis  = over1_2024 - over1_2023
resolved    = over1_2023 - over1_2024

print(f'Communities in deficit (pressure > 1.0):')
print(f'  2023: {len(over1_2023)} SA3s')
print(f'  2024: {len(over1_2024)} SA3s')
print(f'  New in 2024 (crossed threshold): {len(new_crisis)} communities')
print(f'  Resolved (dropped below 1.0):    {len(resolved)} communities')
print()
print('Communities that became crisis zones in 2024:')
new_detail = m24[m24['sa3_name'].isin(new_crisis)].sort_values('waitlist_pressure', ascending=False)
print(new_detail[['sa3_name', 'state', 'mmm_code', 'waitlist_pressure']].round(3).to_string(index=False))

Communities in deficit (pressure > 1.0):
  2023: 48 SA3s
  2024: 58 SA3s
  New in 2024 (crossed threshold): 14 communities
  Resolved (dropped below 1.0):    4 communities

Communities that became crisis zones in 2024:
                      sa3_name state mmm_code  waitlist_pressure
     Daly - Tiwi - West Arnhem    NT      NaN              1.220
            Wheat Belt - South    WA      MM5              1.139
             Campbelltown (SA)    SA      MM1              1.122
             Murray and Mallee    SA      MM5              1.095
                       Sunbury   VIC      MM1              1.068
                   Onkaparinga    SA      MM1              1.065
               Darebin - South   VIC      MM1              1.063
        Gippsland - South West   VIC      MM4              1.063
                      Brighton   TAS      MM2              1.062
   Tablelands (East) - Kuranda   QLD      MM4              1.031
                 Charles Sturt    SA      MM1              1.022
 

### What this means for the trajectory

In a single year, **14 new communities crossed the 1.0 threshold** — meaning demand now structurally exceeds supply in places it did not a year ago. Only 4 communities resolved their deficit. The net change is +10 crisis communities in 12 months.

The 14 new entrants are geographically spread — NT, WA, SA, VIC, TAS, QLD — and span all remoteness bands from MM1 cities (Sunbury VIC, Campbelltown SA, Dandenong VIC) to MM4-5 rural areas (Gippsland South West, Wheat Belt South). This is not a regional problem migrating — it is a national pattern expanding.

**For businesses and workers:** a community crossing 1.0 for the first time represents peak entry opportunity — demand is established and growing, but competition for beds and workers has not yet responded. Communities that crossed in 2024 are today where Noosa Hinterland was two years ago.

## What This Means for Each Audience

---

### For Families Planning Aged Care

If your region appears in this list, **start the ACAT assessment process now, not when crisis hits.** The waitlist in regions like Noosa Hinterland (2.83) or Sunshine Coast Hinterland (2.62) means that by the time a relative is assessed as needing residential care urgently, the queue is already years long.

**Practical step:** Use this chapter's table to check your region's pressure score. If it is above 1.5, build a transition plan that includes relocation options.

---

### For Businesses and Investors Identifying Market Gaps

The 58 SA3 regions in structural deficit are not markets to avoid — they are markets where demand is documented, growing, and unmet. In regions like Wheat Belt North WA (pressure 2.61) and Sunshine Coast Hinterland QLD (pressure 2.62), high-needs home care users already outnumber residential beds by more than 2:1.

For operators evaluating new facilities or expansions, `waitlist_pressure > 1.0` combined with `pop_65_plus > 5,000` identifies the highest-opportunity markets — places where a new facility will fill immediately because the queue already exists.

**Practical step:** Cross-reference `waitlist_pressure > 1.0` with `n_residential < 5` in `service_supply_by_sa3.csv` to identify high-demand, low-competition SA3s.

---

### For Workers Choosing Where to Work

115 SA3 regions saw a decline in residential facilities between 2019 and 2024. These are not markets that are saturated — they are markets where operators have exited, leaving behind documented, growing, unmet demand.

For workers, high-pressure regions mean job security. Places like Taree-Gloucester NSW (pressure 1.71, 766 beds) and Tullamarine-Broadmeadows VIC (pressure 1.78, 707 beds) are large facilities under sustained demand — not small struggling operators.

**Practical step:** Cross-reference `waitlist_pressure > 1.0` with `supply_change < 0` to find regions where demand is rising while supply is contracting — the strongest signal of sustained workforce need.

---

## Data Limitation Note

Indigenous and NESB demographic breakdown is only available at ACPR level (73 regions) — it cannot be joined to SA3 (358 regions). This chapter does not include demographic equity analysis at the SA3 level. If demographic equity is a priority, a separate ACPR-level analysis is required, clearly labelled as a different geographic unit.

## Visual G — Beds Being Built in the Wrong Places: Supply Divergence 2019–2024

Do crisis zones lose residential beds while non-crisis zones gain them?

Compare total residential places 2019→2024 for:
- **Crisis zones** — the 58 SA3s where `waitlist_pressure > 1.0` in 2024
- **Rest of Australia** — the remaining SA3s with adequate supply

Both lines indexed to 100 at 2019 for direct comparison.

Source: `service_supply_by_sa3.csv` + `service_users_by_sa3.csv`

In [ ]:
# Identify crisis SA3s from 2024 snapshot (already built in df)
crisis_sa3s = set(df[df['waitlist_pressure'] > 1.0]['sa3_code'])

# Tag supply rows
supply_tagged = supply.copy()
supply_tagged['is_crisis'] = supply_tagged['sa3_code'].isin(crisis_sa3s)

# Total residential places by year and group (cap at 2024 — pop data only goes to 2024)
yr_grp = (
    supply_tagged[supply_tagged['year'] <= 2024]
    .groupby(['is_crisis', 'year'])['residential_places']
    .sum()
    .reset_index()
)

# Index to 100 at 2019 for comparison
base_2019 = yr_grp[yr_grp['year'] == 2019].set_index('is_crisis')['residential_places']
yr_grp['indexed'] = yr_grp.apply(
    lambda r: r['residential_places'] / base_2019[r['is_crisis']] * 100, axis=1
)
yr_grp['group'] = yr_grp['is_crisis'].map({
    True:  'Crisis zones (waitlist pressure > 1.0)',
    False: 'Rest of Australia'
})

fig_g = px.line(
    yr_grp.sort_values('year'),
    x='year', y='indexed', color='group',
    markers=True,
    color_discrete_map={
        'Crisis zones (waitlist pressure > 1.0)': '#d62728',
        'Rest of Australia':                       '#2ca02c'
    },
    labels={
        'indexed': 'Residential places (2019 = 100)',
        'year':    'Year',
        'group':   ''
    },
    title='Beds being built in the wrong places — supply divergence 2019–2024<br>'
          '<sup>58 crisis SA3s lost beds while the rest of Australia grew supply by 6%</sup>',
    height=430,
)
fig_g.add_hline(y=100, line_width=1, line_dash='dash', line_color='grey',
                annotation_text='2019 baseline', annotation_position='bottom right')
fig_g.update_layout(plot_bgcolor='white', xaxis=dict(tickmode='linear', dtick=1))
fig_g.show()

# Raw numbers
crisis_2019 = supply_tagged[supply_tagged['is_crisis'] & (supply_tagged['year'] == 2019)]['residential_places'].sum()
crisis_2024 = supply_tagged[supply_tagged['is_crisis'] & (supply_tagged['year'] == 2024)]['residential_places'].sum()
non_2019    = supply_tagged[(~supply_tagged['is_crisis']) & (supply_tagged['year'] == 2019)]['residential_places'].sum()
non_2024    = supply_tagged[(~supply_tagged['is_crisis']) & (supply_tagged['year'] == 2024)]['residential_places'].sum()

print(f'Crisis zones:       {crisis_2019:.0f} → {crisis_2024:.0f} places  ({crisis_2024 - crisis_2019:+.0f})')
print(f'Rest of Australia:  {non_2019:.0f} → {non_2024:.0f} places  ({non_2024 - non_2019:+.0f})')

### Insight — Visual G: Supply Divergence

**The beds being built are going to communities that already have enough.**

`[data — service_supply_by_sa3.csv + service_users_by_sa3.csv, 2019–2024]`

| Group | Places 2019 | Places 2024 | Change |
|-------|-------------|-------------|--------|
| 58 crisis SA3s (pressure > 1.0) | 26,587 | 26,036 | **−551** |
| Rest of Australia | 189,402 | 201,429 | **+12,027** |

The 58 communities where demand structurally exceeds supply collectively **lost 551 residential places** between 2019 and 2024. Over the same period, the rest of Australia added 12,027 places — a 6% increase.

This is not a failure of investment in aggregate. Nationally, beds are being built. The failure is **geographic**: new supply is flowing to areas that already have adequate capacity, bypassing the communities most in need.

Combined with the demand story — L3+L4 home care users in crisis zones grew from 140,968 to 172,285 nationally (+22%) — the result is a double squeeze: demand surged while supply fell in exactly the same places.

> **For families:** If your region has been in deficit for years and nothing has changed, this explains why. Supply investment is not going there — it is going to regions already above the line.

> **For investors:** The 58 crisis zones represent a structural gap — documented demand, contracting supply, no new entrants. Regions like Noosa Hinterland (2.83 pressure), Wheat Belt North WA (2.61), and Sunshine Coast Hinterland (2.62) have waiting queues with no supply response.

> **Dashboard callout (error):** Crisis zones lost 551 residential places between 2019 and 2024 while the rest of Australia added 12,027. New supply is going to regions that already have enough — not to the 58 communities where demand has outrun capacity.